# Data Description
- Background: Unit 1.4授權取得官方圖資及metadata；本包保存12個成功原始回應供來源／CRS查證，不替換本機圖資、不重作既有raw文件。
- date received: 20261126（本次實際取得；UTC逐檔記於manifest）
- Path to data file: data.taipei指定都市計畫資料頁的主計／細計下載資源，以及uptilev.gov.taipei官方GIS服務、臺北市法規及都發局頁；精確URL見包內source_manifest.json。
- Unit of observation: ZIP成員為原始官方回應；兩個內層ZIP各有6個SHP／SHX／DBF配套，JSON為服務metadata或少量診斷特徵，HTML為來源頁。
- Sample period: 官方資料頁說明內容更新至114年1月；主細計資源更新2026-04-09，服務為2026-11-26取得的當前回應。不是2024／2025獨立年度圖資。
- Known issues: 主／細計ZIP均無PRJ／CPG／XML／README；GIS服務顯示CRS3857但sourceSpatialReference為3826，不直接證明下載檔CRS；服務筆數／欄位與下載不同。未成功取得的參考請求只留失敗紀錄，不作CRS證據。
- Definition for each variable: manifest的member為包內精確檔名，source_url／response_url為原始及回應網址，version為來源發布／內容標籤，retrieved_at_utc為實際取得時間，bytes／sha256由回應bytes計算；HTTP日期不是計畫生效日。ArcGIS的spatialReference為服務／回應坐標，sourceSpatialReference為服務宣告來源參考，query的outSR是請求輸出而非原CRS證明。
- Data path: `data/raw/unit14_urban_planning_evidence.zip`；[證據包](unit14_urban_planning_evidence.zip)
- Role: `raw`（原始證據包，不是正式processed圖資）
- Verification status: 依下方實際執行結果；逐成員hash、配套清單、metadata及原始樣本核對。
- SHA-256: `695e15f7230c076b2b827f72b4337273ff795f6fc7e9c52680641c972dd72730`
- Next processing notebook: [都市計畫來源及CRS驗證](../code/taipei_urban_planning_crs_validation.ipynb)

Author：詹明翰、謝子詮、曾翊涵（沿用已確認共同作者，不推定個別分工）。建立／修改日期2026-11-26。外層ZIP為本地保存容器，12個原始官方回應bytes未更動，並非政府發布的單一ZIP。


In [1]:
from pathlib import Path
from datetime import datetime,timezone,date
import hashlib,json,io,zipfile,sys,os,platform,struct
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
def find_root():
    for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
        if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():return p
    raise FileNotFoundError('Open within student project containing AGENTS.md and templates/.')
ROOT=find_root();TEMP=ROOT/'data/temp/unit14_validation';TEMP.mkdir(parents=True,exist_ok=True)
assert Path(sys.executable).resolve()==(ROOT/'.venv/Scripts/python.exe').resolve()
assert sys.prefix!=sys.base_prefix
os.environ['PROJ_NETWORK']='OFF'
def file(relative,area='data',must_exist=True):
    rel=Path(relative)
    if rel.is_absolute() or '..' in rel.parts:raise ValueError('Exact project-relative path required')
    p=(ROOT/rel).resolve()
    if not p.is_relative_to((ROOT/area).resolve()):raise ValueError('Path outside intended project area')
    if must_exist and not p.is_file():raise FileNotFoundError(relative)
    return p
def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def verify(expected):
    rows=[]
    for relative,digest in expected.items():
        p=file(relative);assert sha(p)==digest,f'Input changed; stop and inspect version: {relative}'
        rows.append({'path':relative,'bytes':p.stat().st_size,'sha256':digest})
    return pd.DataFrame(rows)
def show(title,table,block=20):
    display(Markdown('**'+title+'**'))
    if not len(table):display(Markdown('無符合紀錄。'))
    for start in range(0,len(table),block):display(table.iloc[start:start+block])
pd.set_option('display.max_colwidth',110)
print({'python':platform.python_version(),'project_venv':True,'network_in_notebook':'none'})

EXPECTED_SHA256={'data/raw/unit14_urban_planning_evidence.zip':'695e15f7230c076b2b827f72b4337273ff795f6fc7e9c52680641c972dd72730'}
show('來源包hash',verify(EXPECTED_SHA256))


{'python': '3.12.2', 'project_venv': True, 'network_in_notebook': 'none'}


**來源包hash**

,path,bytes,sha256
0,data/raw/unit14_urban_planning_evidence.zip,6177243,695e15f7230c076b2b827f72b4337273ff795f6fc7e9c52680641c972dd72730


## Change for Your Own Data
本檔只讀來源包；內層ZIP與JSON在記憶體檢查，未解壓覆寫raw，不連網。


In [2]:
archive=zipfile.ZipFile(file('data/raw/unit14_urban_planning_evidence.zip'))
manifest=json.loads(archive.read('source_manifest.json'))
assert len(manifest)==12
assert set(archive.namelist())=={r['member'] for r in manifest}|{'source_manifest.json','failed_requests.json'}
for row in manifest:
    payload=archive.read(row['member'])
    assert len(payload)==row['bytes'] and hashlib.sha256(payload).hexdigest()==row['sha256']
show('官方原始回應、版本、URL與hash',pd.DataFrame(manifest),block=4)
show('未成功請求紀錄；不當作官方內容',pd.DataFrame(json.loads(archive.read('failed_requests.json'))))
members=[]
for plan,name in [('細計','official_detailed_plan.zip'),('主計','official_main_plan.zip')]:
    z=zipfile.ZipFile(io.BytesIO(archive.read(name)))
    for i in z.infolist():members.append({'plan':plan,'member':i.filename,'bytes':i.file_size,'zip_member_time':str(i.date_time),
                'sha256':hashlib.sha256(z.read(i.filename)).hexdigest()})
    assert not any(i.filename.lower().endswith(('.prj','.cpg','.xml','.md','.txt')) for i in z.infolist())
print({'official_plan_archives':2,'CRS_definition_sidecars_in_plan_archives':0,'archive_is_local_preservation_container':True})


**官方原始回應、版本、URL與hash**

,member,source_url,version,retrieved_at_utc,bytes,sha256,response_url,content_type,http_last_modified,status
0,dataset_page.html,https://data.taipei/dataset/detail?id=3bab0a01-7936-4218-8cb5-f74dfcb43dda,dataset metadata viewed 2026-11-26,2026-11-26T06:48:42.027953+00:00,85871,a13992dae629b7520cd6c78e88f9ed6b87f900e16b01eddcfbdffe98bd4663d9,https://data.taipei/dataset/detail?id=3bab0a01-7936-4218-8cb5-f74dfcb43dda,text/html;charset=utf-8,,NaN
1,official_detailed_plan.zip,https://data.taipei/api/frontstage/tpeod/dataset/resource.download?rid=0cb6e68a-87c1-4846-b766-bcde3ebe179c,全市細計; resource updated 2026-04-09 11:49:14; page describes content through 114年1月,2026-11-26T06:49:06.900307+00:00,4187541,d89e3ab3dcdc60de2b2b83987c0774d87ae3037d53af2adf5a28f47b3a44a21b,https://data.taipei/api/frontstage/tpeod/dataset/resource.download?rid=0cb6e68a-87c1-4846-b766-bcde3ebe179c,application/zip;charset=UTF-8,,NaN
2,official_main_plan.zip,https://data.taipei/api/frontstage/tpeod/dataset/resource.download?rid=10196e7d-2460-4b8a-b1d2-84001d09d7a4,全市主計; resource updated 2026-04-09 11:49:21; page describes content through 114年1月,2026-11-26T06:49:25.265064+00:00,2008868,7abe625f329454e1079bc449d3c71f54c7e58eeafecdb71eb2e64f88f3c7f146,https://data.taipei/api/frontstage/tpeod/dataset/resource.download?rid=10196e7d-2460-4b8a-b1d2-84001d09d7a4,application/zip;charset=UTF-8,,NaN
3,udd_webgis_index.html,https://webgis.udd.gov.taipei/,"official GIS application; current capture, not certified as the same source geometry",2026-11-26T06:49:25.416525+00:00,137,6c9520fba79d49c630383cb9c9360f2bc6afc7bae1ef0f437dab61d1aa85b227,https://webgis.udd.gov.taipei/,text/html,"Sat, 27 Jul 2024 16:38:14 GMT",NaN


,member,source_url,version,retrieved_at_utc,bytes,sha256,response_url,content_type,http_last_modified,status
4,UrbanPlan2_MapServer.json,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer?f=pjson,current service metadata; linkage to local SHP not certified,2026-11-26T06:50:16.026074+00:00,2384,2310eacb6c1b7b1e444b4d3187b64492cf04a86148c9095cb8adadd0d763af29,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer?f=pjson,text/plain;charset=UTF-8,,retrieved
5,UrbanPlan2_layer0.json,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/0?f=pjson,current annotation layer metadata,2026-11-26T06:50:16.126208+00:00,4634,bd5c7683710df876a85b45e7020fb00e0c076d9e23ea95d9a8a91395ac3623ae,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/0?f=pjson,text/plain;charset=UTF-8,,retrieved
6,UrbanPlan2_layer2.json,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/2?f=pjson,current large-scale zoning layer metadata,2026-11-26T06:50:20.166449+00:00,98165,1c383562c2dbeb7fe47f41bca3eeb5c3be1d8e11908149d29fde50a716e58975,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/2?f=pjson,text/plain;charset=UTF-8,,retrieved
7,planning_law_chapter2.html,https://laws.gov.taipei/law/LawSearch/LawChapterArticle/FL003591?chapterno=2,"official published legal text, articles 15/22 as planning-level context only",2026-11-26T06:50:25.141530+00:00,60012,3e579cb215a6faa518cadf65c8ce58c33cc204fcdde7482f1fa58f09216251be,https://laws.gov.taipei/law/LawSearch/LawChapterArticle/FL003591?chapterno=2,text/html; charset=utf-8,,retrieved


,member,source_url,version,retrieved_at_utc,bytes,sha256,response_url,content_type,http_last_modified,status
8,udd_overall_review.html,https://udd.gov.taipei/events/f3gxcu9-10720,official plan review announcements; not a historical citywide SHP archive,2026-11-26T06:50:27.377899+00:00,64411,43473cc63689ca98759fb403f785ee2e4df431d967ba33257f6c6880ea4e22e6,https://udd.gov.taipei/events/f3gxcu9-10720,text/html; charset=utf-8,,retrieved
9,layer2_count.json,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/2/query?where=1%3D1&return...,"current service; query inputs recorded in URL; outSR3826 is requested output, not source-CRS proof",2026-11-26T06:51:30.322289+00:00,15,949cf8bfaff08e201caccca39eebb09fecef5245ae6970c6d60ce0ee96f80729,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/2/query?where=1%3D1&return...,application/json;charset=UTF-8,NaN,retrieved
10,layer2_controls.json,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/2/query?where=%E7%B7%A8%E8...,"current service; query inputs recorded in URL; outSR3826 is requested output, not source-CRS proof",2026-11-26T06:51:30.497962+00:00,16419,62104e9f4c532d430e99e37e6ef806e574dc5f08a4984f8562a075bd15f75fb2,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/2/query?where=%E7%B7%A8%E8...,application/json;charset=UTF-8,NaN,retrieved
11,layer0_controls.json,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/0/query?where=%E7%B7%A8%E8...,"current service; query inputs recorded in URL; outSR3826 is requested output, not source-CRS proof",2026-11-26T06:51:30.626538+00:00,1363,045ea75bcc3277ed0d50973072c7cc0be83dacb216aff538006d2e4cb9522e67,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/0/query?where=%E7%B7%A8%E8...,application/json;charset=UTF-8,NaN,retrieved


**未成功請求紀錄；不當作官方內容**

,member,source_url,version,attempted_at_utc,status,error,record_source
0,national_upload_manual112.pdf,https://ngis.tcd.gov.tw/UpLoadData/國家地理資訊系統都市計畫圖資上傳作業手冊112年版.pdf,112年版 general upload guidance; not proof of local CRS,2026-11-26T06:50:23.459511+00:00,failed,AssertionError: national_upload_manual112.pdf,NaN
1,NaN,https://uptilev.gov.taipei/arcgis/rest/services/UrbanPlan2/UrbanPlan2/MapServer/0/query?where=%E5%88%86%E5...,NaN,NaN,failed,ArcGIS error 400: Unable to complete operation.,Unit 1.4 execution log; failed body was not archived and is not used as CRS evidence


{'official_plan_archives': 2, 'CRS_definition_sidecars_in_plan_archives': 0, 'archive_is_local_preservation_container': True}


## Summary
### Sample
配套清單實際前十列，以及官方少量查詢原始attributes；不假設其編號是跨版本穩定ID。


In [3]:
show('原始配套清單前十列',pd.DataFrame(members).head(10))
for name in ['layer2_controls.json','layer0_controls.json']:
    obj=json.loads(archive.read(name))
    show(name+'原始屬性前十列',pd.DataFrame([f['attributes'] for f in obj.get('features',[])[:10]]))


**原始配套清單前十列**

,plan,member,bytes,zip_member_time,sha256
0,細計,細計-文.dbf,3121,"(2026, 4, 9, 10, 39, 50)",eba92881fc11e574f24024380c946d73cf213e86040663c4070d15b1d552a7be
1,細計,細計-文.shp,184,"(2026, 4, 9, 10, 39, 50)",fa1767fc78c69b7683ec53e0fab8937bac6c33868bedee3c3dcc2ed38dcab731
2,細計,細計-文.shx,124,"(2026, 4, 9, 10, 39, 50)",32d642964d494f49d08830f9c12857f9f4995b56eebf09ef2be994034b864760
3,細計,細計-面.dbf,13982136,"(2026, 4, 9, 10, 39, 50)",29b0484feef66c6c8417821ada8a28ed1edc0efcdffb492476fe7ada8739e1d4
4,細計,細計-面.shp,7556848,"(2026, 4, 9, 10, 39, 50)",67d2ecc469383459e43f2609cf384a4129e09b27c590cbbb82dfa750eba758ee
5,細計,細計-面.shx,124244,"(2026, 4, 9, 10, 39, 50)",277e5f623c5c81865aa7288fac979a279b340899a4b3bc4335638f797fc5fd69
6,主計,主計圖-文.dbf,3445842,"(2026, 4, 9, 10, 39, 22)",e8c8d9f3dca9090be6e7defc68acbf387484212d1be9e739e4cab0e121417b2c
7,主計,主計圖-文.shp,107172,"(2026, 4, 9, 10, 39, 22)",6f9cfae833170e475b7c00275709fad568d8e7688ab531034c4aa24f2ab545d9
8,主計,主計圖-文.shx,30692,"(2026, 4, 9, 10, 39, 22)",3374ff95b3e87bfa4e3892c10f2c0a07c73d7badf91cf376105734131df0c481
9,主計,主計圖-面.dbf,3460258,"(2026, 4, 9, 10, 39, 20)",a8c7eef033837d998672368a8266409d6f461dea6cb73cd810ce175223c910f9


**layer2_controls.json原始屬性前十列**

,OBJECTID,編號,圖層,分區代碼,分區簡稱,使用分區
0,15393,100,39,PM,市,市場用地
1,16293,1000,32,PEB,綠,綠地用地
2,20289,5000,5,R3,住3,第三種住宅區
3,25289,10000,5,R3,住3,第三種住宅區
4,30284,15000,49,PW1,河川區,河川區


**layer0_controls.json原始屬性前十列**

,OBJECTID,編號,圖層,分區代碼,分區簡稱,使用分區
0,15294,1,70,PEA,公,公園用地
1,15295,2,70,PEA,公,公園用地
2,15296,3,70,PEA,公,公園用地


### Summary Stats
服務來源CRS、服務顯示CRS與下載ZIP的CRS證據分開。


In [4]:
show('全部12個官方圖資配套',pd.DataFrame(members),block=12)
map_meta=json.loads(archive.read('UrbanPlan2_MapServer.json'))
rows=[]
for name in ['UrbanPlan2_layer0.json','UrbanPlan2_layer2.json']:
    obj=json.loads(archive.read(name))
    rows.append({'member':name,'layer':obj['name'],'geometryType':obj['geometryType'],
                 'display_spatialReference':str(obj['extent']['spatialReference']),
                 'sourceSpatialReference':str(obj.get('sourceSpatialReference')),'fields':len(obj['fields'])})
show('服務metadata實際值',pd.DataFrame(rows))
print({'service_display_spatialReference':map_meta['spatialReference'],
       'service_polygon_count':json.loads(archive.read('layer2_count.json'))['count'],
       'downloaded_SHP_CRS_confirmed_by_this_document':False})
verify(EXPECTED_SHA256)
print({'all_12_responses_unchanged':True,'existing_raw_files_modified':False})


**全部12個官方圖資配套**

,plan,member,bytes,zip_member_time,sha256
0,細計,細計-文.dbf,3121,"(2026, 4, 9, 10, 39, 50)",eba92881fc11e574f24024380c946d73cf213e86040663c4070d15b1d552a7be
1,細計,細計-文.shp,184,"(2026, 4, 9, 10, 39, 50)",fa1767fc78c69b7683ec53e0fab8937bac6c33868bedee3c3dcc2ed38dcab731
2,細計,細計-文.shx,124,"(2026, 4, 9, 10, 39, 50)",32d642964d494f49d08830f9c12857f9f4995b56eebf09ef2be994034b864760
3,細計,細計-面.dbf,13982136,"(2026, 4, 9, 10, 39, 50)",29b0484feef66c6c8417821ada8a28ed1edc0efcdffb492476fe7ada8739e1d4
4,細計,細計-面.shp,7556848,"(2026, 4, 9, 10, 39, 50)",67d2ecc469383459e43f2609cf384a4129e09b27c590cbbb82dfa750eba758ee
5,細計,細計-面.shx,124244,"(2026, 4, 9, 10, 39, 50)",277e5f623c5c81865aa7288fac979a279b340899a4b3bc4335638f797fc5fd69
6,主計,主計圖-文.dbf,3445842,"(2026, 4, 9, 10, 39, 22)",e8c8d9f3dca9090be6e7defc68acbf387484212d1be9e739e4cab0e121417b2c
7,主計,主計圖-文.shp,107172,"(2026, 4, 9, 10, 39, 22)",6f9cfae833170e475b7c00275709fad568d8e7688ab531034c4aa24f2ab545d9
8,主計,主計圖-文.shx,30692,"(2026, 4, 9, 10, 39, 22)",3374ff95b3e87bfa4e3892c10f2c0a07c73d7badf91cf376105734131df0c481
9,主計,主計圖-面.dbf,3460258,"(2026, 4, 9, 10, 39, 20)",a8c7eef033837d998672368a8266409d6f461dea6cb73cd810ce175223c910f9


**服務metadata實際值**

,member,layer,geometryType,display_spatialReference,sourceSpatialReference,fields
0,UrbanPlan2_layer0.json,文字註記,esriGeometryPoint,"{'wkid': 102100, 'latestWkid': 3857}","{'wkid': 102443, 'latestWkid': 3826}",17
1,UrbanPlan2_layer2.json,都市計畫使用分區2(大比例尺),esriGeometryPolygon,"{'wkid': 102100, 'latestWkid': 3857}","{'wkid': 102443, 'latestWkid': 3826}",18


{'service_display_spatialReference': {'wkid': 102100, 'latestWkid': 3857}, 'service_polygon_count': 15511, 'downloaded_SHP_CRS_confirmed_by_this_document': False}
{'all_12_responses_unchanged': True, 'existing_raw_files_modified': False}
